# Landing → Bronze

Ingestão das 8 tabelas num notebook só (mesma lógica pra todas: ler fiel à origem, escrever com
`overwrite`) — muda só por PR revisado, igual ao notebook base em `analises/`. No workspace do grupo
este notebook fica em `Shared/case-final-rocket-lab/medalhao/Landing_To_Bronze`.

Leitura 100% `spark.read` (padrão Databricks). O grupo está ingerindo direto da pasta do Workspace
(`Shared/case-final-rocket-lab/dados`) usando o esquema `file:` no caminho — ex.:
`file:/Workspace/Shared/case-final-rocket-lab/dados/base_motivos.csv` — e isso funciona pra leitura
distribuída (testado pelo grupo). Esse notebook segue o mesmo padrão: o widget `base_path` aceita tanto
uma pasta do Workspace (prefixo `file:` é adicionado automaticamente) quanto um Volume do Unity Catalog
(`/Volumes/...`, sem prefixo — funciona nativamente, sem precisar de `file:`).

Metadados de origem (`_source_file`, `_source_file_mod_ts`) vêm da coluna oculta `_metadata` do próprio
arquivo, sem precisar escrever o nome à mão.

⚠️ **Catálogo/schema/sufixo de destino** (`CATALOGO`, `SCHEMA_BRONZE`, `SUFIXO` na célula de
configuração) seguem o exemplo que já estava em uso para uma tabela (`dados_vcredit.default.base_motivos_raw`)
— **confirmar com o grupo que é esse o padrão combinado pras 8 tabelas** antes de rodar em produção;
é só mudar essas 3 constantes se decidirem outro catálogo/schema.

Regras da camada bronze (ver `CLAUDE.md`):
- **Fiel à origem**: tudo lido como `string` (`inferSchema=false`), nenhuma correção de conteúdo aqui
  (isso é papel da silver).
- **Idempotente**: escreve com `overwrite` — rodar de novo dá o mesmo resultado.
- **Rastreável**: toda tabela ganha `_source_file`, `_source_file_mod_ts` e `_ingestion_ts`.
- **Sem dado pessoal em output**: a verificação no final só mostra contagens, nunca `display()`/`show()`
  de linhas (⚠️ `clientes` tem CPF/nome/e-mail).

## 1. Configuração

In [ ]:
dbutils.widgets.text("base_path", "")
BASE_PATH = dbutils.widgets.get("base_path")
if not BASE_PATH:
    raise ValueError(
        "Preencha o widget 'base_path'. Exemplos: "
        "'/Workspace/Shared/case-final-rocket-lab/dados' (pasta do Workspace, como o grupo está usando) "
        "ou '/Volumes/<catalogo>/<schema>/<volume>' (Volume do Unity Catalog, se migrarmos pra lá)."
    )

# pasta do Workspace precisa do esquema "file:" pra leitura distribuída funcionar; Volume não precisa
_PREFIXO = "" if BASE_PATH.startswith("/Volumes") else "file:"

# Destino das tabelas bronze — AJUSTAR se o grupo combinar outro catálogo/schema/sufixo.
# Valores abaixo seguem o exemplo já em uso para base_motivos (dados_vcredit.default.base_motivos_raw).
CATALOGO = "dados_vcredit"
SCHEMA_BRONZE = "default"
SUFIXO = "_raw"

# cria o schema de destino se ainda não existir, pra não falhar na primeira execução
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{SCHEMA_BRONZE}")

## 2. Fontes
Arquivo → (tabela de destino, encoding). `clientes.csv` e `Chamados_Hora.CSV` são `ISO-8859-1`; os
demais `UTF-8`.

In [ ]:
SOURCES = {
    "chamados.csv":            ("chamados",            "UTF-8"),
    "Chamados_Hora.CSV":       ("chamados_hora",       "ISO-8859-1"),
    "clientes.csv":            ("clientes",            "ISO-8859-1"),
    "custos.csv":              ("custos",              "UTF-8"),
    "pesquisa_satisfacao.csv": ("pesquisa_satisfacao", "UTF-8"),
    "base_atendentes.csv":     ("base_atendentes",     "UTF-8"),
    "base_motivos.csv":        ("base_motivos",        "UTF-8"),
    "canais.csv":              ("canais",              "UTF-8"),
}

## 3. Leitura (`spark.read`, fiel à origem) e escrita idempotente na bronze

`inferSchema=false` sempre: deixar o Spark adivinhar o tipo é arriscado (ex.: `id_cliente` é CPF com
zero à esquerda — se virar número, perde o zero e corrompe o dado). O caminho de cada arquivo usa
`file:{BASE_PATH}/{arquivo}` quando `base_path` é pasta do Workspace (ex.:
`file:/Workspace/Shared/case-final-rocket-lab/dados/base_motivos.csv`, igual ao que o grupo já testou),
ou `{BASE_PATH}/{arquivo}` direto quando é Volume. `_source_file`/`_source_file_mod_ts` vêm da coluna
oculta `_metadata` do arquivo lido.

In [ ]:
from pyspark.sql import functions as F

resumo = []

for arquivo, (tabela, encoding) in SOURCES.items():
    leitor = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("encoding", encoding)
    )
    if arquivo != "Chamados_Hora.CSV":  # Chamados_Hora.CSV não tem aspas
        leitor = leitor.option("quote", '"').option("escape", '"')

    df_raw = leitor.load(f"{_PREFIXO}{BASE_PATH}/{arquivo}")

    # padroniza nomes de coluna (remove BOM/espaços à toa, minúsculo) — não muda o conteúdo das células
    for antigo in df_raw.columns:
        novo = antigo.replace("\ufeff", "").strip().lower()
        if novo != antigo:
            df_raw = df_raw.withColumnRenamed(antigo, novo)

    df_bronze = (
        df_raw
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file"),
            F.col("_metadata.file_modification_time").alias("_source_file_mod_ts"),
        )
        .withColumn("_ingestion_ts", F.current_timestamp())
    )

    destino = f"{CATALOGO}.{SCHEMA_BRONZE}.{tabela}{SUFIXO}"
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")               # idempotente: roda de novo, mesmo resultado
        .option("overwriteSchema", "true")
        .saveAsTable(destino)
    )

    linhas = spark.table(destino).count()
    resumo.append((tabela, arquivo, encoding, linhas))
    print(f"{destino}: {linhas:,} linhas")

## 4. Verificação

Contagens esperadas (arquivos originais, ver `docs/achados_analise_geral.md`): chamados 134.474 ·
chamados_hora 131.838 · clientes 47.705 · custos 131.069 · pesquisa_satisfacao 81.021 ·
base_atendentes 25 · base_motivos 14 · canais 4. Essa célula só mostra contagens — nunca linhas
inteiras (⚠️ `clientes` tem CPF/nome/e-mail).

In [ ]:
display(spark.createDataFrame(resumo, ["tabela", "arquivo", "encoding", "linhas"]))

## Checklist antes de comitar / dar push

- [ ] `CATALOGO`/`SCHEMA_BRONZE`/`SUFIXO` conferidos com o grupo (seção 1).
- [ ] Contagens da seção 4 batem com o esperado (`docs/achados_analise_geral.md`).
- [ ] Nenhuma célula de output mostra linha de `clientes` (CPF/nome/e-mail).
- [ ] Rodei duas vezes e o resultado foi o mesmo (idempotente).
- [ ] Se precisou mudar algo, foi por PR revisado — não direto na `main`.